# CUDA host compiler review: cuda-samples PR457

Target: https://github.com/NVIDIA/cuda-samples/pull/457
Base: 5443602d89ed99aede2e4b7bf329daddeadb320e
Head: 70c867aaf00bcca8729c6a422c00c2d07ffb8500

The unconditional option causes a default-build regression: when CMAKE_CUDA_HOST_COMPILER is unset, the generated command contains -ccbin= and nvcc fails to preprocess host compiler properties.

Executed the actual vectorAddDrv generate_fatbin_vectorAdd target in Colab: CUDA13.0.88, CMake3.31.10, GNU13.3.0, architecture75, T4 runtime. These are compilation checks, not GPU execution or performance measurements.

| Revision | Default compiler | Explicit /usr/bin/g++ |
| --- | --- | --- |
| Base | Pass | Pass, custom command ignores selection |
| PR head | Fail: empty -ccbin | Pass, option forwarded |
| Head plus conditional patch | Pass | Pass, option forwarded |

The conditional patch was tested for one sample. The same unconditional expression occurs in all eight changed files, but the other seven targets, Windows/QNX, paths containing spaces and the original Fedora/GCC16 environment were not executed. No full-suite or merged-fix claim.

Run build-probe.py in a fresh Linux CUDA environment, then fix-probe.py in the same Python namespace. The scripts use /content/samples457 and pin both revisions. Fresh build directories are required for repeats because cached fatbins may skip nvcc. PR457-Colab.ipynb embeds the scripts. JSON files retain commands, exit codes and raw stdout/stderr. nvcc-probe.py is a supplementary direct compiler check.

conditional-host-compiler.patch is against the pinned PR head. This is independent review evidence and a proposed adjustment to another contributor's build fix.


In [ ]:
import pathlib, subprocess, json
p=pathlib.Path('/content/compiler-probe.cu')
p.write_text('__global__ void kernel() {}\n')
rows=[]
for flag in [[], ['-ccbin='], ['-ccbin=/usr/bin/g++']]:
 a=['nvcc','-arch=sm_75','-fatbin',str(p),'-o','/content/compiler-probe.fatbin']+flag
 r=subprocess.run(a,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
 rows.append({'args':a,'exit':r.returncode,'output':r.stdout})
print(json.dumps(rows))


In [ ]:
import pathlib, subprocess, json, os
root=pathlib.Path('/content/samples457')
root.mkdir(exist_ok=True)
logs=[]
def run(args):
 r=subprocess.run(args,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
 logs.append({'args':args,'exit':r.returncode,'output':r.stdout})
 return r
repo=root/'repo'
if not repo.exists(): run(['git','clone','--quiet','https://github.com/NVIDIA/cuda-samples.git',str(repo)])
run(['git','-C',str(repo),'fetch','--quiet','origin','pull/457/head'])
for label,rev in [('base','5443602d89ed99aede2e4b7bf329daddeadb320e'),('head','70c867aaf00bcca8729c6a422c00c2d07ffb8500')]:
 run(['git','-C',str(repo),'checkout','--quiet',rev])
 for mode in ['default','explicit']:
  b=root/(label+'-'+mode)
  args=['cmake','-S',str(repo/'cpp/0_Introduction/vectorAddDrv'),'-B',str(b),'-DCMAKE_CUDA_ARCHITECTURES=75','-DCMAKE_MODULE_PATH='+str(repo/'cmake')]
  if mode=='explicit': args+=['-DCMAKE_CUDA_HOST_COMPILER=/usr/bin/g++']
  c=run(args)
  if c.returncode==0: run(['cmake','--build',str(b),'--target','generate_fatbin_vectorAdd','--verbose'])
run(['nvcc','--version'])
run(['cmake','--version'])
run(['g++','--version'])
(root/'build-results.json').write_text(json.dumps(logs,indent=2))
print(json.dumps(logs))


In [ ]:
# Run after build-probe.py; test a conditional host-compiler argument.
import difflib
cm=repo/'cpp/0_Introduction/vectorAddDrv/CMakeLists.txt'
before=cm.read_text()
after=before.replace('add_custom_command(', 'set(CUDA_HOST_COMPILER_ARGS)\nif(CMAKE_CUDA_HOST_COMPILER)\n    list(APPEND CUDA_HOST_COMPILER_ARGS "-ccbin=${CMAKE_CUDA_HOST_COMPILER}")\nendif()\n\nadd_custom_command(', 1).replace('-ccbin=${CMAKE_CUDA_HOST_COMPILER}', '${CUDA_HOST_COMPILER_ARGS}', 1)
# Replace only the original command argument, keeping the quoted list element.
after=before.replace('add_custom_command(', 'set(CUDA_HOST_COMPILER_ARGS)\nif(CMAKE_CUDA_HOST_COMPILER)\n    list(APPEND CUDA_HOST_COMPILER_ARGS "-ccbin=${CMAKE_CUDA_HOST_COMPILER}")\nendif()\n\nadd_custom_command(', 1)
after=after.replace('-Wno-deprecated-gpu-targets -ccbin=${CMAKE_CUDA_HOST_COMPILER}', '-Wno-deprecated-gpu-targets ${CUDA_HOST_COMPILER_ARGS}')
cm.write_text(after)
patch=''.join(difflib.unified_diff(before.splitlines(True),after.splitlines(True),fromfile='a/cpp/0_Introduction/vectorAddDrv/CMakeLists.txt',tofile='b/cpp/0_Introduction/vectorAddDrv/CMakeLists.txt'))
(root/'conditional-host-compiler.patch').write_text(patch)
logs=[]
for mode in ['default','explicit']:
 b=root/('fixed-'+mode)
 args=['cmake','-S',str(cm.parent),'-B',str(b),'-DCMAKE_CUDA_ARCHITECTURES=75','-DCMAKE_MODULE_PATH='+str(repo/'cmake')]
 if mode=='explicit': args+=['-DCMAKE_CUDA_HOST_COMPILER=/usr/bin/g++']
 c=run(args)
 if c.returncode==0: run(['cmake','--build',str(b),'--target','generate_fatbin_vectorAdd','--verbose'])
run(['nvidia-smi'])
print(json.dumps({'logs':logs,'patch':patch}))
